# LibAR 책등 탐지 모델 학습 (YOLO26n 파인튜닝)

**사용법:** 이 파일을 [Google Colab](https://colab.research.google.com)에 업로드 → 런타임 유형을 **T4 GPU**로 변경 → 위에서부터 차례로 실행

**데이터셋:** Roboflow `ains-workspace-boytv / Book spline detection` (약 1.4k장, 인스턴스 세분화). 2번 셀에 스니펫 값 반영 완료.

**학습 전략 (PRD FR-1.2):** ① 공개 데이터셋으로 1차 학습(이 노트북) → ② 대림도서관 촬영 프레임을 추가해 2차 파인튜닝(같은 노트북 재사용)

**⚠ 보안:** 2번 셀의 `api_key`는 개인 비밀값. 노트북을 공모전 제출물(데이터·모델 1식)이나 공개 저장소에 넣기 전에 반드시 지우거나 [Colab Secrets](https://colab.research.google.com)로 대체할 것.

In [ ]:
# 1. 설치
%pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()  # GPU(T4) 인식 확인

In [ ]:
# 2. 데이터셋 다운로드 (Roboflow 스니펫)
# ★ 아래 api_key에 본인 Roboflow 키를 넣으세요 (공개 저장소이므로 키를 커밋하지 말 것)
#   Colab 사용 시 좌측 🔑 Secrets에 ROBOFLOW_API_KEY 저장 후 아래처럼 불러오는 것을 권장:
#   from google.colab import userdata; API_KEY = userdata.get("ROBOFLOW_API_KEY")
# ※ project()에는 표시명이 아니라 URL 슬러그를 넣을 것
#   (주소창 app.roboflow.com/ains-workspace-boytv/★슬러그★/browse)
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"   # ← 본인 키로 교체 (커밋 금지)
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("ains-workspace-boytv").project("book-spline-detection-8ksc8")
dataset = project.version(1).download("yolo26")
print("데이터 위치:", dataset.location)

In [ ]:
# 3. 학습 (YOLO26n, T4에서 약 30분~1시간)
# 이 데이터셋은 인스턴스 세분화(다각형 라벨). 우리 파이프라인은 박스만 쓰므로 '검출'로 학습.
# → Ultralytics가 다각형에서 박스를 자동 도출하여 detect 학습. (라벨 형식 오류 시 아래 seg로 전환)
from ultralytics import YOLO

model = YOLO("yolo26n.pt")          # 검출. 실패 시 "yolo26n-seg.pt"(세분화) 또는 "yolo11n.pt"(폴백)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project="libar",
    name="spine_v1",
)

In [ ]:
# 4. 검증 — PRD 목표: mAP@0.5 >= 0.85
metrics = model.val()
print(f"mAP@0.5      = {metrics.box.map50:.3f}  (목표 0.85)")
print(f"mAP@0.5:0.95 = {metrics.box.map:.3f}")

In [ ]:
# 5. 내보내기 & 다운로드 (best.pt = 파이프라인 교체용, ONNX = 브라우저용)
onnx_path = model.export(format="onnx", imgsz=640)
print("ONNX:", onnx_path)

# 학습 결과 실제 경로를 코드로 자동 확보 (Colab 버전에 따라 runs/detect/... 하위일 수 있음)
best_pt = str(model.trainer.best)          # 예: /content/runs/detect/libar/spine_v1/weights/best.pt
print("best.pt:", best_pt)

from google.colab import files
files.download(best_pt)
files.download(str(onnx_path))

## 다운로드한 모델을 샘플 파이프라인에 연결

`best.pt` 를 `libar-sample/` 폴더에 넣고, `pipeline.py` 의 `detect_books_yolo` 에서 모델 목록 맨 앞에 추가:

```python
for name in ("best.pt", "yolo26n.pt", "yolo11n.pt"):
```

파인튜닝 모델은 클래스가 'book'이 아니라 데이터셋 클래스명이므로, `book_ids` 필터는 `if not book_ids: book_ids = list(model.names)` 처럼 전체 허용으로 완화 (파이프라인에 이미 반영 예정).

## 2차 파인튜닝 (대림 촬영분 확보 후)

1. Roboflow에 새 프로젝트 생성 → 대림 서가 영상 업로드 → 프레임 추출(1~2fps) → Auto Label로 반자동 라벨링
2. 공개 데이터셋과 병합(Roboflow에서 두 데이터 소스를 한 프로젝트로) 후 이 노트북 재실행
3. 이때 3번 셀의 시작 가중치를 `yolo26n.pt` 대신 1차 학습의 `best.pt`로 지정하면 이어서 학습